# 🚧 DrowsyGuard AI — Speed Bump Training (Kaggle)

Train a **single-class speed-bump detector** and export `bump_model.pt`.

### Before running
1. **Settings → Accelerator:** `GPU T4 x2`
2. **Settings → Internet:** `On`
3. **Add Input:** the dataset *Urban Road Hazard Dataset (YOLO Format)*

### Then
`Run All` → wait until the last cell prints `bump_model.zip` → download it from the **Output** panel.


In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import random, shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/bump_dataset")
VAL_FRACTION, SEED = 0.15, 42
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Set this manually (0/1/2/3) only if auto-detection cannot find the bump class.
BUMP_ID_OVERRIDE = None


def parse_names(path):
    """Read classes.txt / *.names -> list of class names in id order."""
    names = []
    for line in path.read_text(encoding="utf-8-sig", errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue
        head, sep, tail = line.partition(":")
        if sep and head.strip().isdigit():
            names.append(tail.strip())
            continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2 and parts[0].isdigit():
            names.append(parts[1].strip())
        else:
            names.append(line)
    return names


# 1) find the speed-bump class id
bump_id, bump_name = BUMP_ID_OVERRIDE, None
class_files = sorted(list(INPUT.rglob("classes.txt")) + list(INPUT.rglob("*.names")))
for path in class_files:
    names = parse_names(path)
    print("classes:", path.name, "->", names)
    if bump_id is None:
        for i, name in enumerate(names):
            if "bump" in name.lower() or "speed" in name.lower():
                bump_id, bump_name = i, name
                break
    if bump_id is not None:
        break

if bump_id is None:
    raise SystemExit(
        "Could not find the bump class in classes.txt.\n"
        "Look at the printed class list, then set BUMP_ID_OVERRIDE at the top of this cell."
    )
print("BUMP CLASS:", bump_id, bump_name)

# 2) pair every image with its label (labels usually sit next to the image)
label_index = {}
for lbl in INPUT.rglob("*.txt"):
    label_index.setdefault(lbl.stem, []).append(lbl)


def label_for(image):
    same = image.with_suffix(".txt")
    if same.exists():
        return same
    parts = image.parts
    if "images" in parts:
        pos = len(parts) - 1 - parts[::-1].index("images")
        candidate = Path(*parts[:pos], "labels", *parts[pos + 1:]).with_suffix(".txt")
        if candidate.exists():
            return candidate
    candidates = label_index.get(image.stem, [])
    return candidates[0] if candidates else None


samples = []
for image in INPUT.rglob("*"):
    if image.suffix.lower() not in IMAGE_EXTS:
        continue
    label = label_for(image)
    if label is None:
        continue
    kept = []
    for line in label.read_text(encoding="utf-8-sig", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(float(parts[0]))
        except ValueError:
            continue
        if class_id == bump_id:
            kept.append("0 " + " ".join(parts[1:5]))
    if kept:
        samples.append((image, kept))

print("images with speed bumps:", len(samples))
assert samples, "No speed-bump labels found - check BUMP_ID_OVERRIDE"

# 3) split and copy (single class -> id 0)
random.seed(SEED)
random.shuffle(samples)
n_val = max(1, int(len(samples) * VAL_FRACTION))
splits = {"val": samples[:n_val], "train": samples[n_val:]}

for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

for split, items in splits.items():
    for i, (image, lines) in enumerate(items):
        stem = f"{split}_{i:06d}"
        shutil.copy2(image, OUT / "images" / split / (stem + image.suffix.lower()))
        (OUT / "labels" / split / (stem + ".txt")).write_text(
            "\n".join(lines) + "\n", encoding="utf-8")

class_name = (bump_name or "speed_bump").strip().replace(" ", "_")
(OUT / "data.yaml").write_text(
    f"path: {OUT.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: {class_name}\n",
    encoding="utf-8")

print("train:", len(splits["train"]), " val:", len(splits["val"]))
print((OUT / "data.yaml").read_text())


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(OUT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    project="/kaggle/working/runs",
    name="bump",
    exist_ok=True,
)


In [ ]:
import shutil, zipfile
from pathlib import Path
from ultralytics import YOLO

best = Path("/kaggle/working/runs/bump/weights/best.pt")
target = Path("/kaggle/working/bump_model.pt")
assert best.exists(), f"best.pt not found at {best} - did training finish?"
shutil.copy2(best, target)
print("saved:", target, round(target.stat().st_size / 1e6, 2), "MB")

metrics = YOLO(str(target)).val(data=str(OUT / "data.yaml"), imgsz=640)
print(metrics.results_dict)

zip_path = Path("/kaggle/working/bump_model.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(target, "bump_model.pt")
print("Download from the Output panel:", zip_path)
